<a href="https://colab.research.google.com/github/comet-toolkit/comet_training/blob/main/curepy_example_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explore basic functionality of the curepy tool for use in harmonisation.


## Objectives:

* Get familiar with the [**curepy**](link-to-docs) tool.
* Understand how to set up curepy to perform retrievals.
* Propagate uncertainties through a basic inverse problem.


**curepy** (Comet Uncertainties for REtrievals in PYthon) is a software package within the CoMet toolkit that can be used to propagate uncertainties and correlations through inverse problems.

Consider a measurement function `f()`, where,

``y = f(x,b)``

`y` is a variable that can be measured, `x` is a vector of parameters that we want to retrieve (sometimes called the state vector), and `b` is a vector of inputs to the measurement function that can be measured.

The measurement function maps the state space to the measurement space. We know the measurement function so it is easy to map the state space to the measurement space, this is what we know as forward propagation. However, if we know the values of the measurements, `y`, and want to calculate the values of the state vector, `x`, that would result in our measured values of `y`, we need a method of inverting `f()`, and propagting uncertainties through the inversion process, this is our inverse problem.


## *Step 1* - Set up the environment

First, install the curepy package.

In [ ]:
import xarray as xr
import numpy as np
from curepy.container.retrieval_input import RetrievalInput
from curepy.retrieval_methods.MCMC import MCMC

## *Step 2* - Read in input data and define measurement function

Next, we read in our data, simulated TOA RadCalNet reflectances for matchups with Sentinel-2 over the GONA RadCalNet site. We then add some noise and a bias to these reflectances, these will act as a proxy for Sentinel-2 measurements.

In [ ]:
#read in simulated RadCalNet data
nc_data = xr.open_dataset(r"C:\Users\jr20\code\RCN_matchup_comparisons\cs_ds_all_matchups_S2_rcn_GONA_geomcorrectedRPV.nc")

#define bias
bias = 0.02

#choose single band
sim_refl = nc_data['sim_reflectance_sensor2_10m_insitu'].loc[{"bands_sensor1_10m_insitu": 1}].values

#define simulated sentinel-2 reflectances
noisy_refl = 0.05 * np.random.normal(0, 1, len(sim_refl)) * sim_refl + sim_refl + bias


We also define our measurement function, for this example we are assuming our 'Sentinel-2' reflectances are offset from our RadCalNet reflectances by a bias term. Using **curepy**, we will retrieve this bias.

In [ ]:
def meas_func(bias, rcn_refl):
    s2_refl = rcn_refl + bias
    return s2_refl 

## *Step 3* - Define `RetrievalInput`

All curepy retrieval methods need a ``RetrievalInput`` object as their input. This object contains all the data and parameters needed to set up the retrieval problem. In this example, we are using the ``build_retrieval_inputs`` function to build the ``RetrievalInput``.

The ``RetrievalInput`` is made up of 4 container objects:

* ``Measurement`` - this stores the `y` data, and any associated uncertainties and correlations.
* ``MeasurementFunction`` - this stores the measurement function `f()` and an initial guess of the `x` values.
* ``Prior`` - this stores information about the prior distributions used to constrain the retrieval.
* ``AncillaryParameter`` - this stores any `b` data, and any associated uncertainties and correlations.

These containers can also be instantiated individually (eg. `` meas = Measurement()``) using individual build functions (eg. ``inp.build_measurement()`` ). The individual objects can then be combined into a ``RetrievalInput`` object. However, it is recommended to use ``build_retrieval_inputs``.

In [ ]:
#instantiate RetrievalInput object
inp = RetrievalInput()

#input data into object to build containers
inp.build_retrieval_inputs(measurement_func=meas_func,
                           initial_guess=0.05,
                           y=noisy_refl,
                           u_y_total=0.05*noisy_refl,
                           corr_y='rand',
                           prior_shape = ["normal"],
                           prior_params=[{"mu": 0, "sigma": 0.1}],
                           b = [sim_refl],
                           u_b = [sim_refl * 0.01],
                           corr_b = [np.eye(len(sim_refl))],
                           b_MC_steps = 1,
                           )

## *Step 4* - Define retrieval method object

Next, we instantiate our retrieval method object. In this example, we are using the Markov Chain Monte Carlo method (MCMC). Each retrieval method has some inputs that can be set. ``MCMC`` requires a number of walkers (`nwalkers`), the steps each walker will take (`steps`), and the amount of steps at the start of the walk to discard (`burn_in`).

In [ ]:
#instantiate retrieval method object
ret = MCMC(nwalkers = 10,
           steps = 100,
           burn_in = 10)

## *Step 5* - Run retrieval

All retrievals are then run using the same command, ``run_retrieval``, this is a method of all the retrieval object classes such as ``MCMC``. The only input ``run_retrieval`` requires is a ``RetrievalInput`` object. There are several optional inputs such as `return_corr` or `return_samples` that can be set to add correlation matrices or the MC samples to the output.

``run_retrieval`` returns a ``RetrievalResult`` object which stores the retrieved `x` values and associated uncertainties, along with any other variables that have been chosen to be output using the input kwargs.

In [ ]:
#run retrieval
res = ret.run_retrieval(inp)

## *Step 6* - A multi-dimensional example

In the previous example, we only fit the bias to one band. We can set up our inverse problem to fit the bias at every wavelength simultaneously. This allows us to get the correlation between the biases.

We need to change our initial guess to the expected shape of the biases across the bands (here we are fitting 4 bands), we also need to set priors for each of the bias values, if you do not input a prior for a component of `x`, a uniform prior spanning the whole state space will be used.

In [ ]:
#define bias for 4 wavelengths
bias = [0.02, 0.03, 0.02, 0.05]

#read in data for 4 bands
sim_refl = nc_data['sim_reflectance_sensor2_10m_insitu'].values

#define simulated Sentinel-2 data for multiple bands
noisy_refl = 0.05 * np.random.normal(0, 1, sim_refl.shape) * sim_refl + sim_refl + bias

In [ ]:
#instantiate RetrievalInput and input data to build containers
inp_multi = RetrievalInput()
inp_multi.build_retrieval_inputs(measurement_func=meas_func,
                           initial_guess=[[0.05, 0.05, 0.05, 0.05]],
                           y=noisy_refl,
                           u_y_total=0.05*noisy_refl,
                           prior_shape = ["normal"]*4,
                           prior_params=[{"mu": 0, "sigma": 0.1}]*4,
                           b = [sim_refl],
                           u_b = [sim_refl * 0.01],
                           corr_b = [np.eye(len(sim_refl))],
                           b_MC_steps = 1,
                           )

The retrieval method object is invariant to different inputs so we can use the same object if we still wish to use the ``MCMC`` method. 

In [ ]:
#run retrieval 
res_multi = ret.run_retrieval(inp_multi)

## *Step 7* - Other retrieval methods

**curepy** also implements the Optimal Estimation retrieval method in the ``OE`` class, there are no required inputs to this class. The Jacobian of the measurement function with respect to the state vector can be optianally defined if it has been pre-computed by the user.

Try running the retrieval using the ``OE`` method using ``inp_multi`` as your ``RetrievalInput``:

In [ ]:
from curepy.retrieval_methods.optimal_estimation import OE

In [ ]:
#Try running the OE retrieval here!
